In [ ]:
import torch
import torchvision
print("Torch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

In [ ]:
!pip install fastapi pydantic torch torchvision transformers qdrant-client sentence-transformers pillow uvicorn gradio nest_asyncio mongita pyngrok -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 46.0 MB/s eta 0:00:00
   ━

In [ ]:
!ngrok config add-authtoken

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from transformers import CLIPProcessor, CLIPModel
# from transformers import AutoProcessor, AutoModelForVision2Seq
import google.generativeai as genai

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# gemini
genai.configure(api_key='')

# Create the model
generation_config = {
  "temperature": 1,
  "top_p": 0.95,
  "top_k": 20,
  "max_output_tokens": 1500,
  "response_mime_type": "text/plain",
}

gemini_model = genai.GenerativeModel(
  model_name="gemini-1.5-flash",
  generation_config=generation_config,
)

# Load mô hình Qwen2-VL-2B-Instruct cho phân tích
# qwen_model_name = "Qwen/Qwen2-VL-2B-Instruct"
# qwen_processor = AutoProcessor.from_pretrained(qwen_model_name)
# qwen_model = AutoModelForVision2Seq.from_pretrained(qwen_model_name)
# qwen_model.to(device)

# Load mô hình CLIP cho embedding
clip_model_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model = CLIPModel.from_pretrained(clip_model_name)
clip_model.to(device)

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e

In [ ]:
import torch
from PIL import Image

def extract_embedding(image):
    """Trích xuất embedding từ ảnh bằng CLIP."""
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        image_features = clip_model.get_image_features(**inputs)
    return image_features[0].cpu().numpy()

def extract_text_embedding(text):
    """Trích xuất embedding từ văn bản bằng CLIP."""
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True, max_length=77).to(device)
    with torch.no_grad():
        text_features = clip_model.get_text_features(**inputs)
    return text_features[0].cpu().numpy()

# def analyze_image(prompt, images):
#     """Phân tích ảnh và trả về kết quả dạng văn bản bằng Qwen2-VL."""
#     if not images:  # Nếu không có ảnh, chỉ xử lý văn bản
#         inputs = qwen_processor(text=prompt, images=None, return_tensors="pt").to(device)
#     else:
#         inputs = qwen_processor(text=prompt, images=images, return_tensors="pt").to(device)
#     with torch.no_grad():
#         outputs = qwen_model.generate(**inputs, max_length=150)
#     return qwen_processor.decode(outputs[0], skip_special_tokens=True)

def analyze_image(prompt, images):
    """Phân tích ảnh và trả về kết quả dạng văn bản bằng Gemini."""
    if not images:  # Nếu không có ảnh, chỉ xử lý văn bản
        response = gemini_model.generate_content(prompt)
        return response.text
    else:
        # Chuẩn bị nội dung cho Gemini (văn bản + ảnh)
        content = [prompt] + [image for image in images]
        response = gemini_model.generate_content(content)
        return response.text

In [ ]:
from torchvision.utils import save_image
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def try_on(person_img, cloth_img):
    person_img = transform(person_img).unsqueeze(0)
    cloth_img = transform(cloth_img).unsqueeze(0)
    result_img = person_img  # Placeholder
    return result_img.squeeze().cpu().detach().numpy()

In [ ]:
# database.py
from qdrant_client import QdrantClient
from qdrant_client.http import models

# Khởi tạo Qdrant
qdrant_client = QdrantClient(":memory:")
collection_name="clothes"
vector_size = 512

def init_qdrant():
    if not qdrant_client.collection_exists(collection_name):
        qdrant_client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE)
        )

def add_cloth(image_embedding, cloth_id):
    qdrant_client.upsert(
        collection_name=collection_name,
        points=[models.PointStruct(id=cloth_id, vector=image_embedding.tolist())]
    )

def search_similar(embedding, top_k=3):
    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=models.SearchRequest(
            vector=embedding.tolist(),
            limit=top_k
        )
    )
    return [result.id for result in results]

In [ ]:
from mongita import MongitaClientDisk

# Khởi tạo Mongita
mongo_client = MongitaClientDisk()
db = mongo_client["fashion_db"]
clothes_collection = db["clothes"]

def add_cloth_metadata(cloth_id, name, image_path):
    clothes_collection.insert_one({"id": cloth_id, "name": name, "image_path": image_path})

def get_cloth_metadata(cloth_id):
    return clothes_collection.find_one({"id": cloth_id})

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from pydantic import BaseModel
import io
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
import uuid
import os
import base64

TEMP_DIR = "temp_images"
os.makedirs(TEMP_DIR, exist_ok=True)

app = FastAPI()
init_qdrant()

@app.get("/")
async def check_server():
    return {"status": "Server is running"}

@app.post("/add_cloth")
async def add_cloth_data(cloth_img: UploadFile = File(...), name: str = Form("Unnamed")):
    cloth_image = Image.open(io.BytesIO(await cloth_img.read())).convert("RGB")
    image_embedding = extract_embedding(cloth_image)
    cloth_id = str(uuid.uuid4())

    # Lưu ảnh vào thư mục tạm
    image_path = os.path.join(TEMP_DIR, f"{cloth_id}.jpg")
    cloth_image = cloth_image.resize((224, 224))
    cloth_image.save(image_path)

    add_cloth(image_embedding, cloth_id)
    add_cloth_metadata(cloth_id, name, image_path)
    return {"status": "success", "cloth_id": cloth_id}

@app.post("/evaluate_outfit")
async def evaluate_outfit(outfit_img: UploadFile = File(...)):
    outfit_image = Image.open(io.BytesIO(await outfit_img.read())).convert("RGB")
    prompt = "Phân tích outfit trong ảnh này."
    analysis = analyze_image(prompt, [outfit_image])
    return {"analysis": analysis}

@app.post("/suggest_combinations")
async def suggest_combinations(outfit_img: UploadFile = File(...)):
    outfit_image = Image.open(io.BytesIO(await outfit_img.read())).convert("RGB")
    prompt = """Gợi ý các phụ kiện hoặc trang phục phù hợp với outfit trong ảnh này bằng cách điền vào mô tả về loại trang phục phù hợp theo mẫu sau. Hãy đảm bảo trả về đúng định dạng, mỗi mục trên một dòng:
                  Áo: {Viết mô tả về áo phù hợp}
                  Quần: {Viết mô tả về quần phù hợp}
                  Mũ: {Viết mô tả về mũ phù hợp}
                  Phụ kiện đi kèm: {Viết mô tả về phụ kiện phù hợp}
                  """
    suggestions_text = analyze_image(prompt, [outfit_image])
    print("Suggestions Text:", suggestions_text)

    descriptions = {}
    lines = suggestions_text.split("\n")
    for line in lines:
        line = line.strip()
        if line.startswith("Áo:"):
            descriptions["Áo"] = line.replace("Áo:", "").strip()
        elif line.startswith("Quần:"):
            descriptions["Quần"] = line.replace("Quần:", "").strip()
        elif line.startswith("Mũ:"):
            descriptions["Mũ"] = line.replace("Mũ:", "").strip()
        elif line.startswith("Phụ kiện đi kèm:"):
            descriptions["Phụ kiện đi kèm"] = line.replace("Phụ kiện đi kèm:", "").strip()
    print("Descriptions:", descriptions)

    suggestions = []
    for category, desc in descriptions.items():
        print(f"Searching for: {category} - {desc}")
        text_embedding = extract_text_embedding(desc)
        similar_ids = search_similar(text_embedding)
        print(f"Similar IDs for {category}: {similar_ids}")
        for id in similar_ids:
            metadata = get_cloth_metadata(id)
            if metadata:
                metadata["category"] = category
                suggestions.append(metadata)
                print(f"Found metadata: {metadata}")


    output_results = []
    for s in suggestions:
        img_path = s["image_path"]
        if os.path.exists(img_path):
            with open(img_path, "rb") as f:
                img_data = base64.b64encode(f.read()).decode("utf-8")
            output_results.append({
                "name": s["name"],
                "image_data": img_data,
                "category": s["category"]
            })
        else:
            output_results.append({
                "name": s["name"],
                "image_data": None,
                "category": s["category"]
            })
    print("Output Results:", output_results)
    return {"suggestions": output_results}

class FashionQARequest(BaseModel):
    question: str

@app.post("/fashion_qa")
async def fashion_qa(request: FashionQARequest):
    answer = analyze_image(f"Trả lời câu hỏi thời trang: {request.question}", [])
    return {"answer": answer}

@app.post("/try_on")
async def try_on_outfit(person_img: UploadFile = File(...), cloth_img: UploadFile = File(...)):
    person_image = Image.open(io.BytesIO(await person_img.read())).convert("RGB")
    cloth_image = Image.open(io.BytesIO(await cloth_img.read())).convert("RGB")
    # Placeholder cho try-on (cần tích hợp CP-VTON thực tế)
    return {"image": "Placeholder - Implement CP-VTON here"}

class SearchDescription(BaseModel):
    description: str

@app.post("/search_by_description")
async def search_by_description(desc: SearchDescription):
    text_embedding = extract_text_embedding(desc.description)
    similar_ids = search_similar(text_embedding)
    results = [get_cloth_metadata(id) for id in similar_ids if get_cloth_metadata(id)]

    # Trả về danh sách gồm tên và dữ liệu ảnh base64
    output_results = []
    for r in results:
        img_path = r["image_path"]
        if os.path.exists(img_path):
            with open(img_path, "rb") as f:
                img_data = base64.b64encode(f.read()).decode("utf-8")
            output_results.append({"name": r["name"], "image_data": img_data})
        else:
            output_results.append({"name": r["name"], "image_data": None})
    return {"results": output_results}

In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

def run_fastapi():
    uvicorn.run(app, host="0.0.0.0", port=8000)

public_url = ngrok.connect(8000).public_url
print(f"Ngrok URL: {public_url}")

run_fastapi()

Ngrok URL: https://354a-34-169-25-87.ngrok-free.app


INFO:     Started server process [11044]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     116.96.45.44:0 - "POST /tryon/ HTTP/1.1" 404 Not Found
INFO:     116.96.45.44:0 - "POST /tryon/ HTTP/1.1" 404 Not Found
INFO:     116.96.45.44:0 - "POST /tryon/ HTTP/1.1" 404 Not Found
INFO:     116.96.45.44:0 - "POST /evaluate_outfit HTTP/1.1" 200 OK
INFO:     116.96.45.44:0 - "POST /evaluate_outfit HTTP/1.1" 200 OK
INFO:     116.96.45.44:0 - "POST / HTTP/1.1" 405 Method Not Allowed
INFO:     2402:800:61c5:8a62:38da:76a5:f65d:e0ea:0 - "POST /evaluate_outfit HTTP/1.1" 200 OK
Suggestions Text:                   Áo: Áo thun trắng hoặc đen trơn, hoặc áo croptop có họa tiết nhẹ nhàng, chất liệu cotton hoặc linen để tạo cảm giác thoải mái.
                  Quần: Quần jeans ống rộng màu sáng, quần vải kaki màu be hoặc quần culottes chất liệu linen.
                  Mũ: Mũ bucket màu trắng hoặc be, mũ cói vành rộng, hoặc mũ len màu pastel.
                  Phụ kiện đi kèm: Túi đeo chéo nhỏ nhắn, mắt kính gọng kim loại, vòng tay kim loại hoặc dây chuyền mảnh.
Descriptions: {'Áo': 

<ipython-input-47-f7b33712b7aa>:27: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


Similar IDs for Áo: ['74325fa7-6870-4d96-ad15-cee4dcf28678', '96b37966-b330-40e0-9444-b2c005fbf22c']
Found metadata: {'id': '74325fa7-6870-4d96-ad15-cee4dcf28678', 'name': 'áo đen', 'image_path': 'temp_images/74325fa7-6870-4d96-ad15-cee4dcf28678.jpg', '_id': ObjectId('67d3b141ed883f70eb5941b7'), 'category': 'Áo'}
Found metadata: {'id': '96b37966-b330-40e0-9444-b2c005fbf22c', 'name': 'quần', 'image_path': 'temp_images/96b37966-b330-40e0-9444-b2c005fbf22c.jpg', '_id': ObjectId('67d3d175ed883f70eb5941b8'), 'category': 'Áo'}
Searching for: Quần - Quần jeans ống rộng màu sáng, quần vải kaki màu be hoặc quần culottes chất liệu linen.
Similar IDs for Quần: ['96b37966-b330-40e0-9444-b2c005fbf22c', '74325fa7-6870-4d96-ad15-cee4dcf28678']
Found metadata: {'id': '96b37966-b330-40e0-9444-b2c005fbf22c', 'name': 'quần', 'image_path': 'temp_images/96b37966-b330-40e0-9444-b2c005fbf22c.jpg', '_id': ObjectId('67d3d175ed883f70eb5941b8'), 'category': 'Quần'}
Found metadata: {'id': '74325fa7-6870-4d96-ad15

In [ ]:
import gradio as gr
import requests
from PIL import Image
import io
import base64
import os

# URL Ngrok (cập nhật động)
ngrok_url = None

# Kiểm tra kết nối đến server
def check_server_connection():
    if not ngrok_url:
        return False
    try:
        response = requests.get(f"{ngrok_url}/", timeout=5)
        return response.status_code == 200
    except requests.RequestException:
        return False

def evaluate_outfit(outfit_img_path):
    if not check_server_connection():
        return "Không thể kết nối đến server. Vui lòng kiểm tra server FastAPI và cập nhật URL."
    try:
        if not os.path.exists(outfit_img_path):
            return "File ảnh không tồn tại."
        with open(outfit_img_path, "rb") as f_outfit:
            response = requests.post(f"{ngrok_url}/evaluate_outfit",
                                    files={"outfit_img": f_outfit})
            if response.status_code != 200:
                return f"Lỗi server: {response.status_code} - {response.text}"
            data = response.json()
            return data.get("analysis", data.get("error", "Không có phản hồi hợp lệ"))
    except Exception as e:
        return f"Lỗi khi gửi yêu cầu: {e}"

def suggest_combinations(outfit_img_path):
    if not check_server_connection():
        return []
    try:
        if not os.path.exists(outfit_img_path):
            return ["File ảnh không tồn tại."]
        with open(outfit_img_path, "rb") as f_outfit:
            response = requests.post(f"{ngrok_url}/suggest_combinations",
                                    files={"outfit_img": f_outfit})
            if response.status_code != 200:
                return [f"Lỗi server: {response.status_code} - {response.text}"]
            data = response.json()
            suggestions = data.get("suggestions", [])
            gallery_items = []
            for s in suggestions:
                if s.get("image_data"):
                    img_bytes = base64.b64decode(s["image_data"])
                    img = Image.open(io.BytesIO(img_bytes))
                    caption = f"{s['category']}: {s['name']}"
                    gallery_items.append((img, caption))
                else:
                    caption = f"{s['category']}: {s['name']} (Không có ảnh)"
                    gallery_items.append((None, caption))
            return gallery_items
    except Exception as e:
        return [f"Lỗi khi gửi yêu cầu: {e}"]

def fashion_qa(question):
    if not check_server_connection():
        return "Không thể kết nối đến server. Vui lòng kiểm tra server FastAPI và cập nhật URL."
    try:
        response = requests.post(f"{ngrok_url}/fashion_qa", json={"question": question})
        if response.status_code != 200:
            return f"Lỗi server: {response.status_code} - {response.text}"
        data = response.json()
        return data.get("answer", data.get("error", "Không có phản hồi hợp lệ"))
    except Exception as e:
        return f"Lỗi khi gửi yêu cầu: {e}"

def try_on(person_img_path, cloth_img_path):
    if not check_server_connection():
        return "Không thể kết nối đến server. Vui lòng kiểm tra server FastAPI và cập nhật URL."
    try:
        if not os.path.exists(person_img_path) or not os.path.exists(cloth_img_path):
            return "File ảnh không tồn tại."
        with open(person_img_path, "rb") as f_person, open(cloth_img_path, "rb") as f_cloth:
            response = requests.post(f"{ngrok_url}/try_on",
                                    files={"person_img": f_person, "cloth_img": f_cloth})
            if response.status_code != 200:
                return f"Lỗi server: {response.status_code} - {response.text}"
            data = response.json()
            if "image" in data and isinstance(data["image"], str):
                return data["image"]
            return data.get("error", "Không có phản hồi hợp lệ")
    except Exception as e:
        return f"Lỗi khi gửi yêu cầu: {e}"

def search_by_description(description):
    if not check_server_connection():
        return []
    try:
        response = requests.post(f"{ngrok_url}/search_by_description",
                                json={"description": description})
        if response.status_code != 200:
            return [f"Lỗi server: {response.status_code} - {response.text}"]
        data = response.json()
        results = data.get("results", [])
        gallery_items = []
        for r in results:
            if r.get("image_data"):
                img_bytes = base64.b64decode(r["image_data"])
                img = Image.open(io.BytesIO(img_bytes))
                caption = r["name"]
                gallery_items.append((img, caption))
            else:
                caption = f"{r['name']} (Không có ảnh)"
                gallery_items.append((None, caption))
        return gallery_items
    except Exception as e:
        return [f"Lỗi khi gửi yêu cầu: {e}"]

def add_cloth_to_db(cloth_img_path, cloth_name):
    if not check_server_connection():
        return "Không thể kết nối đến server. Vui lòng kiểm tra server FastAPI và cập nhật URL."
    try:
        if not os.path.exists(cloth_img_path):
            return "File ảnh không tồn tại."
        print(f"Sending cloth_name: {cloth_name}")  # Debug để kiểm tra giá trị cloth_name
        with open(cloth_img_path, "rb") as f_cloth:
            response = requests.post(
                f"{ngrok_url}/add_cloth",
                files={"cloth_img": f_cloth},
                data={"name": cloth_name}
            )
            if response.status_code != 200:
                return f"Lỗi server: {response.status_code} - {response.text}"
            data = response.json()
            return f"{data.get('status', data.get('error', 'Không có phản hồi hợp lệ'))} - ID: {data.get('cloth_id', 'N/A')}"
    except Exception as e:
        return f"Lỗi khi gửi yêu cầu: {e}"

# Hàm cập nhật URL từ người dùng
def update_ngrok_url(url):
    global ngrok_url
    ngrok_url = url
    return f"Đã cập nhật URL: {ngrok_url}"

# Tạo giao diện Gradio
with gr.Blocks() as demo:
    gr.Markdown("# AI Fashion Assistant với FastAPI Server")
    with gr.Row():
        ngrok_input = gr.Textbox(label="Nhập Ngrok URL", placeholder="Ví dụ: https://xxxx.ngrok-free.app")
        update_button = gr.Button("Cập nhật URL")
        update_button.click(update_ngrok_url, inputs=ngrok_input, outputs=gr.Textbox(label="Thông báo"))

    with gr.Tab("Evaluate Outfit"):
        outfit_img_input = gr.Image(type="filepath", label="Upload Outfit Image")
        evaluate_button = gr.Button("Evaluate")
        evaluate_output = gr.Textbox(label="Analysis")
        evaluate_button.click(evaluate_outfit, inputs=outfit_img_input, outputs=evaluate_output)

    with gr.Tab("Suggest Combinations"):
        outfit_img_input = gr.Image(type="filepath", label="Upload Outfit Image")
        suggest_button = gr.Button("Suggest")
        suggest_output = gr.Gallery(label="Suggestions", show_label=True, elem_id="gallery", columns=3)
        suggest_button.click(suggest_combinations, inputs=outfit_img_input, outputs=suggest_output)

    with gr.Tab("Fashion Q&A"):
        question_input = gr.Textbox(label="Ask a Question")
        qa_button = gr.Button("Ask")
        qa_output = gr.Textbox(label="Answer")
        qa_button.click(fashion_qa, inputs=question_input, outputs=qa_output)

    with gr.Tab("Try On"):
        person_img_input = gr.Image(type="filepath", label="Upload Person Image")
        cloth_img_input = gr.Image(type="filepath", label="Upload Cloth Image")
        try_on_button = gr.Button("Try On")
        try_on_output = gr.Image(label="Result")
        try_on_button.click(try_on, inputs=[person_img_input, cloth_img_input], outputs=try_on_output)

    with gr.Tab("Search by Description"):
        description_input = gr.Textbox(label="Description")
        search_button = gr.Button("Search")
        search_output = gr.Gallery(label="Search Results", show_label=True, elem_id="search_gallery", columns=3)
        search_button.click(search_by_description, inputs=description_input, outputs=search_output)

    with gr.Tab("Add Cloth to Database"):
        cloth_img_input = gr.Image(type="filepath", label="Upload Cloth Image")
        cloth_name_input = gr.Textbox(label="Cloth Name", value="Unnamed")
        add_button = gr.Button("Add to Database")
        add_output = gr.Textbox(label="Result")
        add_button.click(add_cloth_to_db, inputs=[cloth_img_input, cloth_name_input], outputs=add_output)

# Chạy Gradio
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://16048e0d616a845780.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
